# Comparing Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/02_comparing_strategies.ipynb)

**For:** researchers who have run a first simulation, as in
[01_getting_started.ipynb](01_getting_started.ipynb). **Level:** intermediate.

You will compare strategies the way a simulation study does: on one item bank and one population,
over several replications, by the difference each strategy makes in each replication. You will
compare four estimators, three selectors, four kinds of exposure control and two variable-length stoppers, and read
the metrics that tell them apart: RMSE, bias, test length, exposure and overlap. The notebook shows
one example of each kind of method; the [technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html) list them all.

To run the notebook outside this repository, install catsim with its plot extra, which adds
matplotlib: `pip install 'catsim[plot]'`.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from catsim import CatEngine, ItemBank, SimulationResult, SimulationRunner, plot
from catsim.estimation import EAPEstimator, MAPEstimator, NumericalSearchEstimator, WarmLikelihoodEstimator
from catsim.initialization import FixedPointInitializer
from catsim.irt import THETA_MAX_EXTENDED, THETA_MIN_EXTENDED
from catsim.selection import (
  AStratSelector,
  KLSelector,
  MaxInfoSelector,
  ProgressiveSelector,
  RandomesqueSelector,
  UrrySelector,
)
from catsim.stopping import ConfidenceIntervalStopper, MaxLengthStopper, StandardErrorStopper, StopRule

## One bank, one population, several replications

Every strategy below meets the same `N_ITEMS`-item bank and the same `N_EXAMINEES` examinees, and is
run `N_REPLICATIONS` times. The metrics of one run depend on its random draws, the simulated
responses above all, so one run cannot rank two strategies. `SEED`, the notebook's only seed, is
split into a seed for the bank, one for the population and one for each replication, and every
strategy is run with the same replication seeds
([Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html)). Tests have `TEST_LENGTH` items until the last
section: short tests make the strategies differ more.

`replicate` runs an engine once per replication seed. `compare` prints, for each strategy, the mean
over the replications of its RMSE, bias, test length, highest exposure rate, share of the bank never
given and overlap rate. Its `difference` column is the strategy's RMSE minus the first strategy's,
replication by replication: the mean of those differences, ± their standard error. With five
replications, a difference must be about three standard errors from 0 (2.8, the 95% point of a t
distribution with four degrees of freedom) before these replications can tell it from chance.
`at_edge` counts a run's final estimates at either end of the ability scale.

In [ ]:
SEED = 2026
N_ITEMS = 300
N_EXAMINEES = 120
N_REPLICATIONS = 5
TEST_LENGTH = 10

bank_seed, population_seed, *replication_seeds = np.random.SeedSequence(SEED).spawn(2 + N_REPLICATIONS)
item_bank = ItemBank.generate(N_ITEMS, model="3PL", rng=np.random.default_rng(bank_seed))
true_thetas = np.random.default_rng(population_seed).normal(0, 1, N_EXAMINEES)


def replicate(engine: CatEngine) -> list[SimulationResult]:
  """Run the engine on the population once for each replication seed."""
  runner = SimulationRunner(engine)
  return [runner.run(true_thetas, rng=np.random.default_rng(seed)) for seed in replication_seeds]


def paired(first: np.ndarray, second: np.ndarray) -> str:
  """Format the mean over the replications of second minus first, ± its standard error."""
  difference = second - first
  return f"{difference.mean():+.3f} ± {np.std(difference, ddof=1) / np.sqrt(len(difference)):.3f}"


def compare(studies: dict[str, list[SimulationResult]]) -> None:
  """Print each strategy's metrics, averaged over its replications, and its RMSE difference from the first."""
  first = np.array([result.rmse for result in next(iter(studies.values()))])
  print(f"{'':14} {'RMSE':>6} {'difference':>15} {'bias':>7} {'length':>6} {'top rate':>8} {'unused':>6} overlap")
  for name, results in studies.items():
    rmse = np.array([result.rmse for result in results])
    bias = np.mean([result.bias for result in results])
    length = np.mean([result.test_lengths.mean() for result in results])
    top_rate = np.mean([result.exposure_rates.max() for result in results])
    unused = np.mean([np.mean(result.exposure_rates == 0) for result in results])
    overlaps = [result.overlap_rate for result in results]
    overlap = "None" if None in overlaps else f"{np.mean(overlaps):.2f}"
    print(
      f"{name:14} {rmse.mean():6.3f} {paired(first, rmse):>15}"
      f" {bias:+7.3f} {length:6.1f} {top_rate:8.2f} {unused:6.2f} {overlap}"
    )


def at_edge(result: SimulationResult) -> int:
  """Count the final estimates at either end of the ability scale."""
  thetas = result.final_thetas
  return int(np.count_nonzero(np.isclose(thetas, THETA_MIN_EXTENDED) | np.isclose(thetas, THETA_MAX_EXTENDED)))

## Estimators

Four estimators, each with maximum-information selection: expected a posteriori
([EAP](https://douglasrizzo.com.br/catsim/api/techniques/estimation/eap-estimator.html)) and maximum a posteriori
([MAP](https://douglasrizzo.com.br/catsim/api/techniques/estimation/map-estimator.html)), which combine the responses with a standard normal
prior; maximum likelihood ([ML](https://douglasrizzo.com.br/catsim/api/techniques/estimation/numerical-search-estimator.html)), which uses the
responses alone; and Warm's weighted likelihood ([WLE](https://douglasrizzo.com.br/catsim/api/techniques/estimation/warm-wle.html)), which
corrects the bias of maximum likelihood. Estimates are kept within `THETA_MIN_EXTENDED` and
`THETA_MAX_EXTENDED`, −6 and 6; the last line counts, over all replications, the final estimates at
either end.

In [ ]:
estimators = {
  "EAP": EAPEstimator(),
  "ML": NumericalSearchEstimator(),
  "MAP": MAPEstimator(),
  "WLE": WarmLikelihoodEstimator(),
}
fixed_length = MaxLengthStopper(TEST_LENGTH)
by_estimator = {
  name: replicate(CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(), estimator, fixed_length))
  for name, estimator in estimators.items()
}
compare(by_estimator)
edges = {name: sum(at_edge(result) for result in results) for name, results in by_estimator.items()}
print("final estimates at the edge of the scale:", edges)

EAP and MAP tie: their difference, −0.002 ± 0.004, is within one standard error of 0. ML's RMSE is
0.337 ± 0.032 above EAP's and WLE's 0.093 ± 0.014 above, both far beyond three standard errors. ML
alone ends tests at the edge of the scale, five of them, and each such estimate weighs heavily in
its RMSE. The bias is close to 0 for all four, which hides how each estimator errs at the ends of the
scale; the figure shows it.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharex=True, sharey=True)
for ax, (name, results) in zip(axes, by_estimator.items(), strict=True):
  ax.scatter(results[0].true_thetas, results[0].final_thetas, s=8, alpha=0.6)
  ax.axline((0, 0), slope=1, color="gray", linestyle="--")
  ax.set_title(name)
  ax.set_xlabel("true ability θ")
axes[0].set_ylabel("final estimate θ̂")
plt.tight_layout()
plt.show()

for session in by_estimator["ML"][0].sessions:
  if np.isclose(session.current_theta, THETA_MIN_EXTENDED):
    responses = [int(response) for response in session.responses]
    print(f"ML at the lower edge: true ability {session.true_theta:.2f}, responses {responses}")

Each panel plots the final estimates of replication 0 against the true abilities. The EAP and MAP
clouds are flatter than the diagonal: both estimators underestimate high abilities and overestimate
low ones, pulled toward the mean of their prior, and their errors at the two ends cancel in the
bias. WLE follows the diagonal at both ends but scatters more around it: less bias, more variance,
and in tests this short a larger RMSE. ML scatters most, and one examinee ends at −6 although
two of their ten responses are correct (the line under the figure): the guessing parameter gives a
correct response a probability of at least $c$ at any ability, so when the correct responses look
like guesses, the likelihood keeps rising toward the bottom of the scale. The
[estimation](https://douglasrizzo.com.br/catsim/api/techniques/estimation/index.html) page explains estimates at the edges of the scale.

## Selectors

Three selectors, each with the EAP estimator: maximum information at the current estimate
([max info](https://douglasrizzo.com.br/catsim/api/techniques/selection/max-info-selector.html)); the item whose difficulty is closest to the
estimate ([Urry](https://douglasrizzo.com.br/catsim/api/techniques/selection/urry-selector.html)); and Kullback-Leibler information
([KL](https://douglasrizzo.com.br/catsim/api/techniques/selection/kl-selector.html)), which looks past the current estimate: it weighs each
item over a window of abilities around it, wide at the start of the test and narrowing as items
accumulate. The last line compares the run times of KL and max info.

In [ ]:
selectors = {"max info": MaxInfoSelector(), "Urry": UrrySelector(), "KL": KLSelector()}
by_selector = {
  name: replicate(CatEngine(item_bank, FixedPointInitializer(0.0), selector, EAPEstimator(), fixed_length))
  for name, selector in selectors.items()
}
compare(by_selector)
durations = {name: sum(result.duration for result in results) for name, results in by_selector.items()}
print(f"KL took {durations['KL'] / durations['max info']:.1f} times as long as max info")

Urry's RMSE is 0.102 ± 0.006 above max info's: choosing by difficulty alone ignores the items'
discrimination and guessing, which costs precision on a 3PL bank. It uses more of the bank in
exchange, leaving 42% of it unused against 86%, and two examinees share 25% of their items on
average, against 47%. KL's difference from max info, +0.005 ± 0.010, is within one standard error,
so these replications cannot tell the two apart, and KL's runs take several times as long (the
last line).

## Exposure control

Maximum information gives the best items to almost every examinee, who could then pass them on.
Exposure control spreads the items over more of the bank, at some cost in precision. Four methods,
each with the EAP estimator, against max info from the previous section:

- an exposure cap, `MaxInfoSelector(r_max=R_MAX)`, which passes over an item once it has been given
  to a share `R_MAX` of the examinees tested before;
- [randomesque](https://douglasrizzo.com.br/catsim/api/techniques/selection/randomesque-selector.html) selection, which draws each item at
  random from the five most informative;
- the [progressive](https://douglasrizzo.com.br/catsim/api/techniques/selection/progressive-selector.html) method, which blends a random score
  with information, the weight on information growing from 0 at the first item to 1 at the last;
- [a-stratification](https://douglasrizzo.com.br/catsim/api/techniques/selection/stratified-selector.html), which sorts the bank by
  discrimination into one stratum per item of the test and gives the strata in order, keeping the
  most discriminating items for the end.

The progressive and a-stratified selectors plan their schedule for a test of known length, so their
`test_size` is `TEST_LENGTH`, the length at which the stopper ends the test. A shorter test would
end before the a-stratified selector reaches its most discriminating stratum, and before the
progressive schedule turns purely informative; in a longer one, the a-stratified selector runs out
of strata and raises an error.

In [ ]:
R_MAX = 0.2
exposure_selectors = {
  f"r_max={R_MAX}": MaxInfoSelector(r_max=R_MAX),
  "randomesque": RandomesqueSelector(bin_size=5),
  "progressive": ProgressiveSelector(test_size=TEST_LENGTH),
  "a-stratified": AStratSelector(test_size=TEST_LENGTH),
}
by_exposure = {"max info": by_selector["max info"]} | {
  name: replicate(CatEngine(item_bank, FixedPointInitializer(0.0), selector, EAPEstimator(), fixed_length))
  for name, selector in exposure_selectors.items()
}
compare(by_exposure)

Each method lowers some exposure metric. The cap holds every item's rate at 0.20 and cuts the
overlap from 0.47 to 0.17. Randomesque and progressive selection lower the top rate to 0.70 and
0.60, and the progressive method leaves the least of the bank unused, 45%. The RMSE costs of these
three, 0.018 ± 0.009, 0.029 ± 0.016 and 0.023 ± 0.020, are each within three standard errors of 0:
these replications can neither tell them from no cost nor rank them. a-stratification uses more of
the bank than max info but costs 0.091 ± 0.010 of RMSE, nine standard errors: in a 10-item test,
half the items come from the less discriminating half of the bank. Its top rate stays at 1, as for max info: every test starts at the same estimate, so the least discriminating
stratum gives every examinee the same first item.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, name in zip(axes, ["max info", f"r_max={R_MAX}", "a-stratified"], strict=True):
  plot.item_exposure(by_exposure[name][0].exposure_rates, item_bank=item_bank, par="a", ax=ax, title=name)
plt.tight_layout()
plt.show()

Each point is an item of the bank, at its discrimination, with its exposure rate in replication 0.
Max info gives only items whose discrimination is above about 1.4, the most discriminating at rates
up to 1. The cap holds those rates at 0.2 and spreads the tests over more of the discriminating
items. a-stratification gives items across the whole range of discrimination, among them the item
at rate 1 near $a = 0.8$, the first item of every test.

## Stoppers

The tests so far have a fixed length. A variable-length test ends when a criterion is met, and a
`StopRule` combines several, ending the test when any of them holds
([stopping](https://douglasrizzo.com.br/catsim/api/techniques/stopping/index.html)). Two rules, each with a cap of `MAX_ITEMS` items:

- precision: the standard error is at most `TARGET_SE`;
- classification: the 95% confidence interval of the estimate lies on one side of the cut score
  `CUT`, so the examinee can be classified as above or below it.

`rule.satisfied(engine.context(session))` returns the rule's stoppers that hold at the end of a
session: the counts below tell which criterion ended each test. The share classified correctly is
the share of examinees whose final estimate is on the same side of `CUT` as their true ability,
and the lines after it give the classification rule's paired difference in that share from the
other two. The fixed-length tests of max info are the first row.

In [ ]:
TARGET_SE = 0.4
CUT = 0.0
MAX_ITEMS = 30
rules = {
  "precision": StopRule(StandardErrorStopper(TARGET_SE), MaxLengthStopper(MAX_ITEMS)),
  "classification": StopRule(ConfidenceIntervalStopper([CUT]), MaxLengthStopper(MAX_ITEMS)),
}
engines = {
  name: CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(), EAPEstimator(), rule)
  for name, rule in rules.items()
}
by_stopper = {f"{TEST_LENGTH} items": by_selector["max info"]} | {
  name: replicate(engine) for name, engine in engines.items()
}
compare(by_stopper)

correct = {
  name: np.array([np.mean((result.final_thetas > CUT) == (result.true_thetas > CUT)) for result in results])
  for name, results in by_stopper.items()
}
for name, shares in correct.items():
  print(f"{name}: classified correctly {shares.mean():.3f}")
for name in [f"{TEST_LENGTH} items", "precision"]:
  print(f"classification minus {name}: {paired(correct[name], correct['classification'])}")
for name, rule in rules.items():
  ended_by = Counter(
    " and ".join(type(stopper).__name__ for stopper in rule.satisfied(engines[name].context(session)))
    for result in by_stopper[name]
    for session in result.sessions
  )
  for criteria, count in ended_by.most_common():
    print(f"{name}, ended by {criteria}: {count}")

Both variable-length rules give more items than the fixed 10, and so a lower RMSE: 0.095 ± 0.003
lower for the precision rule, with 18.0 items on average. Of its 600 tests (120 examinees in 5
replications), 590 ended by reaching the target standard error and 9 at the 30-item cap; one
reached the target on its 30th item, so both criteria held. The classification rule ended 361
tests when the interval cleared the cut score and 234 at the cap; 5 cleared it on their 30th item.
For examinees near the cut the decision stays in doubt, and the rule keeps testing them while it
stops the others early. It classifies 90.5% of the examinees correctly, against 87.2% and 87.8% for
the other two, but its paired differences, +0.033 ± 0.023 and +0.027 ± 0.012, are within three
standard errors of 0: these replications cannot show that it classifies better. The overlap rate is
None for variable-length tests, whose lengths differ.

## Where to go next

- The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide is the reference for this
  notebook's tasks, and the [technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html) define each method and say when to
  choose it.
- [03_live_sessions.ipynb](03_live_sessions.ipynb) runs the same engine in an application, one
  response at a time.
- [04_custom_strategies.ipynb](04_custom_strategies.ipynb) writes a strategy of your own.